# Bayesian Epistemic Routing - labelling with StripedHyena-Nous-7B

This notebook runs **step 2 (labelling)** on a Colab GPU: StripedHyena answers each question
using our retrieved passages, and every answer is graded correct / "I don't know" / hallucinated.

**Before you start**
1. *Runtime -> Change runtime type -> GPU*: pick **L4** or **A100**. The free **T4 will not work**
   (StripedHyena needs flash-attn, which needs an RTX-30-series-or-newer class GPU).
2. On your Mac, zip the project (code + knowledge base) and put the zip in Google Drive in a
   folder called `prob_reas`:
   ```
   cd ~/Documents/prob_reas
   zip -r prob_reas_upload.zip *.py requirements*.txt kb -x "kb/wiki_cache.jsonl"
   ```
   Then upload `prob_reas_upload.zip` to **My Drive / prob_reas /**.

Labels are written straight to Drive after every question, so if Colab disconnects,
just run the cells again: it continues where it stopped.

## 1. Check the GPU

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> GPU (L4 or A100)"
major, minor = torch.cuda.get_device_capability()
name = torch.cuda.get_device_name()
mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"{name}: compute capability {major}.{minor}, {mem_gb:.0f} GB")
assert major >= 8, f"{name} is too old for flash-attn (needs compute capability 8.0+). Pick an L4 or A100."
if mem_gb < 20:
    print("WARNING: under 20 GB. The 16-bit model needs ~16 GB plus working memory; use LLM = 'stripedhyena-4bit' below.")

## 2. Connect Google Drive and unpack the project

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, zipfile
DRIVE_DIR = "/content/drive/MyDrive/prob_reas"
ZIP = f"{DRIVE_DIR}/prob_reas_upload.zip"
LABELS_DIR = f"{DRIVE_DIR}/labels"          # results live in Drive, safe across disconnects
assert os.path.exists(ZIP), f"Upload prob_reas_upload.zip to {DRIVE_DIR}/ first"
zipfile.ZipFile(ZIP).extractall("/content/prob_reas")
os.makedirs(LABELS_DIR, exist_ok=True)
%cd /content/prob_reas
!ls

## 3. Install packages (flash-attn usually installs from a prebuilt wheel in about a minute)

In [ ]:
!pip -q install -r requirements.txt -r requirements-gpu.txt
!pip -q install flash-attn --no-build-isolation
!python -m spacy download en_core_web_sm -q
import flash_attn; print("flash-attn", flash_attn.__version__, "OK")

## 4. Smoke test: 5 questions
The first run downloads StripedHyena (~15 GB, a few minutes). Check that the replies look like
real short answers before starting the pilot.

In [ ]:
LLM = "stripedhyena"          # or "stripedhyena-4bit" if the GPU has under 20 GB
!python label_dataset.py --llm {LLM} --limit 5 --out {LABELS_DIR}

## 5. Pilot: 200 questions
The summary at the end shows correct / abstained / hallucinated rates, splits by knowledge-base
coverage and entity popularity, and **how many hours 5,000 questions would take** on this GPU.
Copy that summary back into the chat.

In [ ]:
!python label_dataset.py --llm {LLM} --limit 200 --out {LABELS_DIR}

## 6. (Later) Full run on all questions in kb/questions.jsonl
Only after we have looked at the pilot. Same command without `--limit`; it continues from the pilot.

In [ ]:
# !python label_dataset.py --llm {LLM} --out {LABELS_DIR}

## 7. Get the results
The labels are already in **My Drive / prob_reas / labels /** (a `.jsonl` file plus a
`.profile.json` saying which models produced them). Download that folder to your Mac
into `~/Documents/prob_reas/labels/`; learning the probabilities (step 3) runs on the Mac in seconds.